In [0]:
from pyspark.sql import functions as F

CATALOG = "academy"
SILVER_SCHEMA = "lab4_silver"
TABLE_NAME = "netflix_titles"

SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.{TABLE_NAME}"

In [0]:
silver_df = spark.table(SILVER_TABLE)
first_count = silver_df.count()
print(f"First row count: {first_count}")

In [0]:
silver_after_rerun_df = spark.table(SILVER_TABLE)
after_rerun_count = silver_after_rerun_df.count()
print(f"Before rerun: {first_count}")
print(f"After rerun: {after_rerun_count}")

In [0]:
duplicate_count = (
    silver_after_rerun_df
    .groupBy("show_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

### Checking quality rules

In [0]:
quality_results = (
    spark.table(SILVER_TABLE)
    .select(
        F.sum(
            F.when(
                F.col("show_id").isNull()
                | (F.length(F.trim(F.col("show_id"))) == 0),
                1
            ).otherwise(0)
        ).alias("invalid_show_id"),

        F.sum(
            F.when(
                F.col("title").isNull()
                | (F.length(F.trim(F.col("title"))) == 0),
                1
            ).otherwise(0)
        ).alias("invalid_title"),

        F.sum(
            F.when(
                ~F.col("type").isin("Movie", "TV Show"),
                1
            ).otherwise(0)
        ).alias("invalid_type"),

        F.sum(
            F.when(
                ~F.col("release_year").between(
                    1900,
                    F.year(F.current_date())
                ),
                1
            ).otherwise(0)
        ).alias("invalid_release_year")
    )
)

display(quality_results)

In [0]:
%sql
OPTIMIZE academy.lab4_silver.netflix_titles

In [0]:
count_after_optimize = spark.table(SILVER_TABLE).count()

print(f"Before OPTIMIZE: {after_rerun_count}")
print(f"After OPTIMIZE:  {count_after_optimize}")

In [0]:
%sql 
VACUUM academy.lab4_silver.netflix_titles RETAIN 168 HOURS;

In [0]:
%sql
DESCRIBE HISTORY academy.lab4_silver.netflix_titles;